In [1]:
import glob, os, re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, average_precision_score

def clean_tweet(t):
    t = str(t)
    t = re.sub(r"https?://\S+", " ", t)
    t = re.sub(r"@\w+", " ", t)
    t = re.sub(r"^rt\s+", "", t, flags=re.I)
    t = t.replace("#", " ")
    return re.sub(r"\s+", " ", t).strip()

# ---- HumAID, all events
frames = []
for f in glob.glob("data/raw/humaid/events_set1/**/*.tsv", recursive=True):
    name = os.path.basename(f).lower()
    if not name.endswith(("_train.tsv", "_dev.tsv", "_test.tsv")):
        continue
    d = pd.read_csv(f, sep="\t", quoting=3, on_bad_lines="skip")
    d["event"] = name.rsplit("_", 1)[0]
    frames.append(d)
hu = pd.concat(frames, ignore_index=True)

HU_NON = ["not_humanitarian", "sympathy_and_support",
          "rescue_volunteering_or_donation_effort", "other_relevant_information"]
hu["safety_label"] = hu["class_label"].map(
    lambda l: "hazard" if l == "infrastructure_and_utility_damage" else ("non_hazard" if l in HU_NON else None))
hu["text"] = hu["tweet_text"].apply(clean_tweet)
hu = hu.dropna(subset=["safety_label"])
hu = hu[hu["text"].str.len() > 15].drop_duplicates(subset="text")

# ---- T26, all events
fr = []
for p in glob.glob("data/raw/crisislex/CrisisLexT26/*/*-tweets_labeled.csv"):
    d = pd.read_csv(p, encoding_errors="replace", on_bad_lines="skip")
    d.columns = [c.strip() for c in d.columns]
    d["event"] = os.path.basename(os.path.dirname(p))
    fr.append(d)
t26 = pd.concat(fr, ignore_index=True)

T26_NON = ["Sympathy and support", "Donations and volunteering", "Other Useful Information"]
def t26_label(r):
    if r["Information Type"] == "Infrastructure and utilities":
        return "hazard"
    if r["Information Type"] in T26_NON or r["Informativeness"] == "Not related":
        return "non_hazard"
    return None
t26["safety_label"] = t26.apply(t26_label, axis=1)
t26["text"] = t26["Tweet Text"].apply(clean_tweet)
t26 = t26.dropna(subset=["safety_label"])
t26 = t26[t26["text"].str.len() > 15].drop_duplicates(subset="text")

print("HumAID events:", hu["event"].nunique(), "| tweets:", len(hu))
print("T26 events:", t26["event"].nunique(), "| tweets:", len(t26))

HumAID events: 13 | tweets: 32910
T26 events: 26 | tweets: 17098


In [2]:
EN = set("the of and in to is a for on at with has have are was by from this that it as be".split())
def looks_english(t):
    return sum(w in EN for w in re.findall(r"[a-z']+", t.lower())) >= 2

HU_HOLD = ["maryland_floods_2018", "srilanka_floods_2017"]
T26_HOLD = ["2013_Queensland_floods", "2012_Philipinnes_floods", "2013_Manila_floods"]
T26_EXCLUDE = ["2013_Sardinia_floods"]   # Italian

tests = {
    "HumAID unseen floods":   hu[hu["event"].isin(HU_HOLD)],
    "T26 Queensland":         t26[t26["event"] == "2013_Queensland_floods"],
    "T26 Philippines+Manila": t26[t26["event"].isin(["2012_Philipinnes_floods", "2013_Manila_floods"])],
}

hu_pool = hu[~hu["event"].isin(HU_HOLD)]
t26_pool = t26[~t26["event"].isin(T26_HOLD + T26_EXCLUDE)]

FLOOD_KEYS = ["harvey", "irma", "maria", "matthew", "floods", "idai"]
hu_flood = hu_pool[hu_pool["event"].apply(lambda e: any(k in e for k in FLOOD_KEYS))]
t26_flood = t26_pool[t26_pool["event"].isin(["2013_Alberta_floods", "2013_Colorado_floods"])]
t26_other = t26_pool[~t26_pool["event"].isin(["2013_Alberta_floods", "2013_Colorado_floods"])]
t26_other = t26_other[t26_other["text"].apply(looks_english)]   # rough English filter

A = pd.concat([hu_flood, t26_flood])[["text", "safety_label"]]
B = pd.concat([hu_pool, t26_flood, t26_other])[["text", "safety_label"]]

test_texts = set().union(*[set(d["text"]) for d in tests.values()])
A = A[~A["text"].isin(test_texts)].drop_duplicates("text")
B = B[~B["text"].isin(test_texts)].drop_duplicates("text")

for name, d in [("A: flood events only", A), ("B: all events", B)]:
    print(name, len(d), d["safety_label"].value_counts().to_dict())
for tn, d in tests.items():
    print(tn, len(d), int((d["safety_label"] == "hazard").sum()), "hazard")

A: flood events only 25637 {'non_hazard': 20224, 'hazard': 5413}
B: all events 40289 {'non_hazard': 33550, 'hazard': 6739}
HumAID unseen floods 927 96 hazard
T26 Queensland 773 119 hazard
T26 Philippines+Manila 1078 86 hazard


In [3]:
def fit_eval(train, name, th=0.40):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(
        vec.fit_transform(train["text"]), train["safety_label"])
    hz = list(clf.classes_).index("hazard")
    rows = []
    for tn, d in tests.items():
        y = (d["safety_label"] == "hazard").astype(int)
        p = clf.predict_proba(vec.transform(d["text"]))[:, hz]
        pred = np.where(p >= th, "hazard", "non_hazard")
        pr, rc, f1, _ = precision_recall_fscore_support(
            d["safety_label"], pred, labels=["hazard"], zero_division=0)
        rows.append({"model": name, "test": tn, "precision": round(pr[0], 2),
                     "recall": round(rc[0], 2), "f1": round(f1[0], 2),
                     "avg_precision": round(average_precision_score(y, p), 3)})
    return rows

pd.DataFrame(fit_eval(A, "A: flood events only") + fit_eval(B, "B: all events"))

,model,test,precision,recall,f1,avg_precision
0,A: flood events only,HumAID unseen floods,0.50,0.80,0.61,0.748
1,A: flood events only,T26 Queensland,0.55,0.60,0.57,0.572
2,A: flood events only,T26 Philippines+Manila,0.39,0.16,0.23,0.262
3,B: all events,HumAID unseen floods,0.45,0.89,0.60,0.761
4,B: all events,T26 Queensland,0.58,0.60,0.59,0.601
5,B: all events,T26 Philippines+Manila,0.50,0.19,0.27,0.297


## Findings

- Training sets: A = flood/hurricane events only (25,637 tweets, 5,413 hazard); B = all HumAID and T26 events, with a rough English filter on extra T26 tweets (40,289 tweets, 6,739 hazard). Same unseen test sets as notebook 09; test tweets removed from training. Threshold fixed at 0.40, not re-tuned.
- Average precision, A to B: HumAID unseen floods 0.748 to 0.761, T26 Queensland 0.572 to 0.601, T26 Philippines+Manila 0.262 to 0.297. Small but consistent gains; test sets have only 86 to 119 hazard tweets, so differences of this size are not conclusive.
- Philippines+Manila stays weak (recall 0.19), so training data size is not the main limit there.
- Likely limits (not tested): bag-of-words model, label disagreement between datasets, event type, 2012 to 2019 language.
- Decision: baseline v2 (notebook 09, threshold chosen on dev) stays the official Safety baseline; A/B are a data-size experiment.